# Notebook 19: Cross-Species Alignment

## Evolutionary Neuroscience Meets Foundation Models

**Author**: NeuroS Team  
**Date**: 2025-11-25  
**Duration**: ~30 minutes

---

## Overview

Evolution is nature's optimization algorithm. By comparing neural representations across species, we can identify **conserved computational principles** that transcend specific implementations. This notebook demonstrates:

### Core Topics:

1. **Procrustes Alignment** - Optimal linear transformations between neural spaces
2. **Representational Similarity Analysis (RSA)** - Geometry-based comparisons
3. **Phylogenetic Analysis** - Relating neural similarity to evolutionary distance
4. **Conserved/Specific Decomposition** - Separating shared vs unique features
5. **Foundation Model Applications** - Testing AI alignment with biology

### Why Cross-Species Analysis Matters:

**The Comparative Method** is a cornerstone of evolutionary biology. Applied to neuroscience:

- **Universal principles**: Features conserved across species likely solve fundamental problems
- **Specializations**: Species-specific features reveal adaptations to ecological niches
- **Constraints**: Phylogenetic correlations reveal evolutionary constraints on neural design
- **AI validation**: Do foundation models discover the same solutions as evolution?

### Key References:

- **Kriegeskorte et al. (2008)**: Representational similarity analysis
- **Gower & Dijksterhuis (2004)**: Procrustes problems in statistics
- **Yamins & DiCarlo (2016)**: Using goal-driven models to understand the brain
- **Kriegeskorte & Wei (2021)**: Neural tuning and representational geometry

---

In [ ]:
# Core imports
import numpy as np
import torch
import torch.nn as nn
from typing import Dict, List, Tuple
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr, pearsonr
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Bokeh for interactive visualizations
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import gridplot, column, row
from bokeh.models import HoverTool, Span, Label, ColorBar, LinearColorMapper, Arrow, VeeHead
from bokeh.palettes import Viridis256, Turbo256, RdBu11, Category10, Spectral11
from bokeh.transform import linear_cmap

# NeuroS-MechInt alignment components
from neuros_mechint.alignment import (
    ProcrustesAlignment,
    CrossSpeciesRSA,
    PhylogeneticDistance,
    ConservedSpecificDecomposition
)

# Enable Bokeh in Jupyter
output_notebook()

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

print("✓ All imports successful")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ PyTorch version: {torch.__version__}")

---

## Part 1: Procrustes Alignment

### Mathematical Foundation

**Procrustes analysis** finds the optimal orthogonal transformation to align two point clouds. Given matrices $X$ (source) and $Y$ (target), we seek rotation $R$, scaling $s$, and translation $t$ to minimize:

$$
\min_{R, s, t} \|Y - (sXR + t\mathbf{1}^T)\|_F^2
$$

where $\| \cdot \|_F$ is the Frobenius norm.

**Solution** via singular value decomposition (SVD):
1. Center both matrices: $\tilde{X} = X - \bar{X}$, $\tilde{Y} = Y - \bar{Y}$
2. Compute $M = \tilde{Y}^T \tilde{X}$
3. SVD: $M = U \Sigma V^T$
4. Optimal rotation: $R = UV^T$
5. Optimal scale: $s = \text{tr}(\tilde{Y}^T \tilde{X} R) / \text{tr}(\tilde{X}^T \tilde{X})$

### Simulating Cross-Species Data

In [ ]:
# Simulate neural representations from mouse and macaque V1
n_stimuli = 100  # Natural images
n_dims = 20      # Neural dimensions

# Mouse V1 representation (baseline)
np.random.seed(42)
mouse_V1 = np.random.randn(n_stimuli, n_dims)

# Add structure: some stimuli cluster together (e.g., similar orientations)
cluster_centers = np.random.randn(5, n_dims) * 3
cluster_assignments = np.random.randint(0, 5, n_stimuli)
for i in range(n_stimuli):
    mouse_V1[i] += cluster_centers[cluster_assignments[i]] * 0.5

# Macaque V1: similar but with rotation + scaling + noise
# (simulates evolutionary divergence with conserved function)
angle = np.pi / 4  # 45 degree rotation in first 2 dims
R_true = np.eye(n_dims)
R_true[:2, :2] = np.array([[np.cos(angle), -np.sin(angle)], 
                           [np.sin(angle), np.cos(angle)]])

macaque_V1 = (mouse_V1 @ R_true.T) * 1.3  # Scaled by 1.3
macaque_V1 += np.random.randn(n_stimuli, n_dims) * 0.8  # Add noise

print(f"✓ Generated neural representations")
print(f"  Mouse V1: {mouse_V1.shape}")
print(f"  Macaque V1: {macaque_V1.shape}")
print(f"  Ground truth rotation: 45° in first 2 dimensions")
print(f"  Ground truth scale: 1.3")

In [ ]:
# Perform Procrustes alignment
aligner = ProcrustesAlignment()
aligned_mouse, disparity = aligner.align(mouse_V1, macaque_V1)

print(f"\n✓ Procrustes alignment complete")
print(f"  Disparity (error): {disparity:.4f}")
print(f"  Estimated scale: {aligner.scale:.4f} (true: 1.3)")
print(f"  Rotation matrix: {aligner.R.shape}")

# Project to 2D for visualization using PCA
pca = PCA(n_components=2)
mouse_2d = pca.fit_transform(mouse_V1)
macaque_2d = pca.transform(macaque_V1)
aligned_2d = pca.transform(aligned_mouse)

print(f"  PCA variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
# Interactive Procrustes visualization

# Panel 1: Before alignment
p_before = figure(
    title='Before Alignment',
    width=450,
    height=450,
    x_axis_label='PC1',
    y_axis_label='PC2'
)
p_before.circle(mouse_2d[:, 0], mouse_2d[:, 1], size=8, color='blue', 
                alpha=0.6, legend_label='Mouse V1')
p_before.circle(macaque_2d[:, 0], macaque_2d[:, 1], size=8, color='red', 
                alpha=0.6, legend_label='Macaque V1')
hover_before = HoverTool(tooltips=[('Species', '@legend_label'), ('PC1', '@x{0.00}'), ('PC2', '@y{0.00}')])
p_before.add_tools(hover_before)
p_before.legend.location = 'top_right'

# Panel 2: After alignment
p_after = figure(
    title='After Procrustes Alignment',
    width=450,
    height=450,
    x_axis_label='PC1',
    y_axis_label='PC2',
    x_range=p_before.x_range,
    y_range=p_before.y_range
)
p_after.circle(aligned_2d[:, 0], aligned_2d[:, 1], size=8, color='cyan', 
               alpha=0.6, legend_label='Mouse (aligned)')
p_after.circle(macaque_2d[:, 0], macaque_2d[:, 1], size=8, color='red', 
               alpha=0.6, legend_label='Macaque V1')
hover_after = HoverTool(tooltips=[('Species', '@legend_label'), ('PC1', '@x{0.00}'), ('PC2', '@y{0.00}')])
p_after.add_tools(hover_after)
p_after.legend.location = 'top_right'

# Panel 3: Alignment error (per stimulus)
errors = np.linalg.norm(aligned_mouse - macaque_V1, axis=1)

p_error = figure(
    title='Alignment Error per Stimulus',
    width=900,
    height=300,
    x_axis_label='Stimulus Index',
    y_axis_label='Euclidean Distance'
)
p_error.circle(range(len(errors)), errors, size=6, color='purple', alpha=0.7)
p_error.line(range(len(errors)), errors, line_width=1, color='purple', alpha=0.3)
# Add mean error line
mean_error = Span(location=errors.mean(), dimension='width', 
                 line_color='red', line_dash='dashed', line_width=2)
p_error.add_layout(mean_error)
mean_label = Label(x=5, y=errors.mean()+0.5, text=f'Mean = {errors.mean():.2f}', 
                  text_font_size='10pt', text_color='red')
p_error.add_layout(mean_label)
hover_error = HoverTool(tooltips=[('Stimulus', '@x'), ('Error', '@y{0.00}')])
p_error.add_tools(hover_error)

# Combine
grid_proc = gridplot([[p_before, p_after], [p_error]])
show(grid_proc)

print("\n📊 Procrustes Alignment Visualization")
print("   Top-left: Original representations (misaligned)")
print("   Top-right: After alignment (much closer)")
print("   Bottom: Residual error per stimulus")

---

## Part 2: Representational Similarity Analysis (RSA)

### Representational Dissimilarity Matrices (RDMs)

RSA compares **geometries** rather than individual vectors. For a set of stimuli, the RDM captures pairwise dissimilarities:

$$
D_{ij} = d(\mathbf{r}_i, \mathbf{r}_j)
$$

where $\mathbf{r}_i$ is the neural response to stimulus $i$ and $d$ is a distance metric (often correlation distance: $d = 1 - \text{corr}$).

**Key insight**: Two systems with similar RDMs have similar representational geometries, even if their coordinate systems differ.

### Multi-Species Comparison

In [ ]:
# Generate representations for multiple species
species_names = ['Mouse', 'Rat', 'Ferret', 'Macaque', 'Human']
n_species = len(species_names)

# Shared signal (evolutionarily conserved)
conserved_signal = np.random.randn(n_stimuli, n_dims) * 3

# Generate species-specific representations
species_reps = {}
phylo_distances = [0, 12, 62, 92, 96]  # Million years since common ancestor with mouse

for i, name in enumerate(species_names):
    # More divergence for phylogenetically distant species
    noise_scale = 0.5 + (phylo_distances[i] / 100) * 1.5
    
    species_rep = conserved_signal.copy()
    species_rep += np.random.randn(n_stimuli, n_dims) * noise_scale
    
    # Add species-specific structure
    species_signal = np.random.randn(n_stimuli, n_dims) * 1.5
    species_rep += species_signal * (phylo_distances[i] / 100)
    
    species_reps[name] = species_rep

print(f"✓ Generated representations for {n_species} species")
for name, rep in species_reps.items():
    print(f"  {name}: {rep.shape}")

In [ ]:
# Compute RDMs for each species
rdms = {}
for name, rep in species_reps.items():
    rdm = squareform(pdist(rep, metric='correlation'))
    rdms[name] = rdm

# Compute cross-species RDM similarity
rsa = CrossSpeciesRSA()
similarity_matrix = rsa.compute_similarity_matrix(species_reps)

print(f"\n✓ Computed RDMs and cross-species similarities")
print(f"  RDM shape: {rdms['Mouse'].shape}")
print(f"  Similarity matrix: {similarity_matrix.shape}")

In [ ]:
# Interactive RDM visualization

# Subsample for clarity (show 30x30 instead of 100x100)
subsample = 30
rdm_mouse_sub = rdms['Mouse'][:subsample, :subsample]
rdm_human_sub = rdms['Human'][:subsample, :subsample]

# Panel 1: Mouse RDM
p_rdm_mouse = figure(
    title='Mouse V1 RDM',
    width=450,
    height=450,
    x_axis_label='Stimulus',
    y_axis_label='Stimulus',
    x_range=(0, subsample),
    y_range=(0, subsample)
)

mapper_rdm = LinearColorMapper(palette=Viridis256, low=0, high=2)
p_rdm_mouse.image(image=[rdm_mouse_sub], x=0, y=0, dw=subsample, dh=subsample, 
                 color_mapper=mapper_rdm)
color_bar_m = ColorBar(color_mapper=mapper_rdm, width=15, location=(0, 0),
                      title='Distance')
p_rdm_mouse.add_layout(color_bar_m, 'right')

# Panel 2: Human RDM
p_rdm_human = figure(
    title='Human V1 RDM',
    width=450,
    height=450,
    x_axis_label='Stimulus',
    y_axis_label='Stimulus',
    x_range=(0, subsample),
    y_range=(0, subsample)
)

p_rdm_human.image(image=[rdm_human_sub], x=0, y=0, dw=subsample, dh=subsample, 
                 color_mapper=mapper_rdm)
color_bar_h = ColorBar(color_mapper=mapper_rdm, width=15, location=(0, 0),
                      title='Distance')
p_rdm_human.add_layout(color_bar_h, 'right')

# Panel 3: Cross-species similarity matrix
p_sim = figure(
    title='Cross-Species Representational Similarity',
    width=600,
    height=500,
    x_range=species_names,
    y_range=list(reversed(species_names)),
    toolbar_location=None
)

# Flatten for rect glyph
species_x = []
species_y = []
sim_values = []

for i, s1 in enumerate(species_names):
    for j, s2 in enumerate(species_names):
        species_x.append(s1)
        species_y.append(s2)
        sim_values.append(similarity_matrix[i, j])

# Color mapper
sim_mapper = LinearColorMapper(palette=RdBu11[::-1], 
                               low=similarity_matrix.min(), high=similarity_matrix.max())

p_sim.rect(x=species_x, y=species_y, width=1, height=1,
          fill_color={'field': 'sim', 'transform': sim_mapper},
          line_color='white', line_width=2,
          source={'x': species_x, 'y': species_y, 'sim': sim_values})

# Add text annotations
from bokeh.models import LabelSet, ColumnDataSource
source = ColumnDataSource(data={'x': species_x, 'y': species_y, 
                                'text': [f'{s:.2f}' for s in sim_values]})
labels = LabelSet(x='x', y='y', text='text', source=source,
                 text_align='center', text_baseline='middle', text_font_size='9pt')
p_sim.add_layout(labels)

color_bar_sim = ColorBar(color_mapper=sim_mapper, width=15, location=(0, 0),
                        title='Similarity')
p_sim.add_layout(color_bar_sim, 'right')

p_sim.xaxis.major_label_orientation = 0.785

# Combine
grid_rsa = gridplot([[p_rdm_mouse, p_rdm_human], [p_sim]])
show(grid_rsa)

print("\n📊 RSA Visualization")
print("   Top: RDMs for Mouse and Human (similar patterns = similar geometry)")
print("   Bottom: Cross-species similarity matrix")
print("   \n   Notice: Closely related species have higher similarity")

---

## Part 3: Phylogenetic Analysis

### Mantel Test: Neural vs Phylogenetic Distance

If neural representations are constrained by phylogeny, we expect:

$$
\text{Neural dissimilarity} \propto \text{Phylogenetic distance}
$$

The **Mantel test** assesses this correlation by permuting one matrix and computing correlation under null hypothesis.

### Phylogenetic Tree Construction

In [ ]:
# Define phylogenetic distances (millions of years since divergence)
# Approximate mammalian phylogeny
phylo_dist_matrix = np.array([
    [0,   12,  62,  92,  96],   # Mouse
    [12,  0,   62,  92,  96],   # Rat (diverged ~12 Mya)
    [62,  62,  0,   92,  96],   # Ferret (diverged ~62 Mya)
    [92,  92,  92,  0,   25],   # Macaque (diverged ~92 Mya from rodents, 25 Mya from humans)
    [96,  96,  96,  25,  0 ]    # Human
])

print("✓ Phylogenetic distance matrix defined")
print("\nPairwise divergence times (Mya):")
print("       ", " ".join([f"{s:8s}" for s in species_names]))
for i, s in enumerate(species_names):
    print(f"{s:8s}", " ".join([f"{phylo_dist_matrix[i,j]:6.0f}  " for j in range(n_species)]))

In [ ]:
# Test phylogenetic correlation

# Extract upper triangles (avoid diagonal and redundancy)
triu_idx = np.triu_indices(n_species, k=1)
neural_sim = similarity_matrix[triu_idx]
phylo_dist = phylo_dist_matrix[triu_idx]

# Correlation: neural similarity should decrease with phylogenetic distance
rho, pval = spearmanr(phylo_dist, neural_sim)

print(f"\n✓ Phylogenetic correlation analysis")
print(f"  Spearman ρ = {rho:.3f}")
print(f"  p-value = {pval:.4f}")
print(f"  \n  Interpretation: {'Significant' if pval < 0.05 else 'Not significant'} negative correlation")
print(f"  (Greater phylogenetic distance → Lower neural similarity)")

In [ ]:
# Interactive phylogenetic correlation plot

# Create labels for each pair
pair_labels = []
for i in range(n_species):
    for j in range(i+1, n_species):
        pair_labels.append(f"{species_names[i]}-{species_names[j]}")

# Panel 1: Scatter plot with regression
p_phylo = figure(
    title='Neural Similarity vs Phylogenetic Distance',
    width=700,
    height=500,
    x_axis_label='Phylogenetic Distance (Mya)',
    y_axis_label='Neural Similarity (correlation)'
)

# Points
from bokeh.models import ColumnDataSource
source_phylo = ColumnDataSource(data={
    'x': phylo_dist,
    'y': neural_sim,
    'labels': pair_labels
})

p_phylo.circle('x', 'y', size=12, color='blue', alpha=0.7, source=source_phylo)

# Regression line
from scipy.stats import linregress
slope, intercept, r_value, p_value, std_err = linregress(phylo_dist, neural_sim)
x_fit = np.linspace(phylo_dist.min(), phylo_dist.max(), 100)
y_fit = slope * x_fit + intercept

p_phylo.line(x_fit, y_fit, line_width=2, color='red', alpha=0.8, 
            legend_label=f'R² = {r_value**2:.3f}')

# Add statistics text
stats_label = Label(x=70, y=neural_sim.max()*0.95, 
                   text=f'ρ = {rho:.3f}\np = {pval:.4f}', 
                   text_font_size='11pt',
                   background_fill_color='white',
                   background_fill_alpha=0.8)
p_phylo.add_layout(stats_label)

hover_phylo = HoverTool(tooltips=[('Pair', '@labels'), 
                                  ('Distance', '@x{0.0} Mya'), 
                                  ('Similarity', '@y{0.000}')])
p_phylo.add_tools(hover_phylo)
p_phylo.legend.location = 'bottom_left'

show(p_phylo)

print("\n📊 Phylogenetic Correlation")
print("   Each point: species pair")
print("   X-axis: Time since common ancestor")
print("   Y-axis: Neural representational similarity")
print("   \n   Negative slope confirms phylogenetic constraint")

---

## Part 4: Conserved vs Species-Specific Decomposition

### Shared Response Model (SRM)

Decompose each species' representation into:
- **Conserved subspace**: Shared across all species
- **Species-specific subspace**: Unique to each species

For species $s$ with representation matrix $X^{(s)}$:

$$
X^{(s)} = W^{(s)} S + E^{(s)}
$$

where:
- $S$ is the shared latent space (conserved)
- $W^{(s)}$ is species-specific weights
- $E^{(s)}$ is species-specific residual

### Applying CSD

In [ ]:
# Conserved-Specific Decomposition
csd = ConservedSpecificDecomposition(n_conserved=8, n_specific=5)

# Prepare data
species_list = [species_reps[name] for name in species_names]

# Decompose
result = csd.decompose(species_list)

conserved_components = result['conserved_components']
specific_components = result['specific_components']
conserved_var = result['conserved_variance']
specific_var = result['specific_variance']

print(f"✓ Conserved-Specific Decomposition complete")
print(f"  Conserved components: {conserved_components.shape}")
print(f"  Species-specific components: {len(specific_components)} species")
print(f"\n  Variance explained:")
for i, name in enumerate(species_names):
    total = conserved_var[i] + specific_var[i]
    print(f"    {name:8s}: {conserved_var[i]/total*100:5.1f}% conserved, "
          f"{specific_var[i]/total*100:5.1f}% specific")

In [ ]:
# Interactive CSD visualization

# Panel 1: Variance breakdown
p_var = figure(
    title='Conserved vs Species-Specific Variance',
    width=700,
    height=400,
    x_range=species_names,
    y_axis_label='Variance Explained'
)

# Stack bars
p_var.vbar(x=species_names, top=conserved_var, width=0.7, 
          color='green', alpha=0.7, legend_label='Conserved')
p_var.vbar(x=species_names, top=specific_var, bottom=conserved_var, width=0.7,
          color='orange', alpha=0.7, legend_label='Species-specific')

hover_var = HoverTool(tooltips=[('Species', '@x'), ('Variance', '@top{0.00}')])
p_var.add_tools(hover_var)
p_var.legend.location = 'top_right'
p_var.xaxis.major_label_orientation = 0.785

# Panel 2: Conserved components heatmap
p_cons = figure(
    title='Conserved Components (Top 8)',
    width=600,
    height=400,
    x_axis_label='Original Dimension',
    y_axis_label='Conserved Component',
    x_range=(0, n_dims),
    y_range=(0, 8)
)

cons_mapper = LinearColorMapper(palette=RdBu11[::-1], 
                                low=conserved_components[:, :8].min(), 
                                high=conserved_components[:, :8].max())

p_cons.image(image=[conserved_components[:, :8].T], x=0, y=0, dw=n_dims, dh=8, 
            color_mapper=cons_mapper)
color_bar_cons = ColorBar(color_mapper=cons_mapper, width=15, location=(0, 0),
                         title='Weight')
p_cons.add_layout(color_bar_cons, 'right')

# Panel 3: Phylogenetic relationship to conservation
# More distant species should have lower conservation
mean_phylo_dist = phylo_dist_matrix.mean(axis=1)
conservation_fraction = conserved_var / (conserved_var + specific_var)

p_cons_phylo = figure(
    title='Conservation vs Mean Phylogenetic Distance',
    width=550,
    height=400,
    x_axis_label='Mean Distance to Other Species (Mya)',
    y_axis_label='Conserved Fraction'
)

source_cons = ColumnDataSource(data={
    'x': mean_phylo_dist,
    'y': conservation_fraction,
    'names': species_names
})

p_cons_phylo.circle('x', 'y', size=15, color='purple', alpha=0.7, source=source_cons)

# Add labels
from bokeh.models import LabelSet
labels_cons = LabelSet(x='x', y='y', text='names', x_offset=5, y_offset=5,
                      source=source_cons, text_font_size='9pt')
p_cons_phylo.add_layout(labels_cons)

hover_cons_phylo = HoverTool(tooltips=[('Species', '@names'), 
                                       ('Dist', '@x{0.0} Mya'), 
                                       ('Conservation', '@y{0.00}')])
p_cons_phylo.add_tools(hover_cons_phylo)

# Combine
grid_csd = gridplot([[p_var], [p_cons, p_cons_phylo]])
show(grid_csd)

print("\n📊 Conserved-Specific Decomposition")
print("   Top: Variance breakdown by species")
print("   Bottom-left: Conserved component structure")
print("   Bottom-right: Conservation vs phylogenetic distance")

---

## Part 5: Foundation Model Applications

### Testing Biological Alignment

Do foundation models trained on different datasets discover the same representational geometry as evolution?

**Experiment**:
1. Train models on different visual datasets
2. Extract representations for same stimuli
3. Apply cross-species analysis methods
4. Compare to biological benchmarks

In [ ]:
# Simulate foundation models trained on different datasets
class SimpleVisionModel(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, hidden_dim),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.encoder(x)

# Create models trained on different datasets
model_datasets = {
    'ImageNet': SimpleVisionModel(hidden_dim=128),
    'COCO': SimpleVisionModel(hidden_dim=128),
    'Places365': SimpleVisionModel(hidden_dim=128),
    'iNaturalist': SimpleVisionModel(hidden_dim=128)
}

# Generate test stimuli
test_stimuli = torch.randn(n_stimuli, 784)  # Flattened 28x28 images

# Extract representations
model_reps = {}
with torch.no_grad():
    for dataset_name, model in model_datasets.items():
        model.eval()
        reps = model(test_stimuli).numpy()
        model_reps[dataset_name] = reps

print("✓ Foundation model representations extracted")
for name, reps in model_reps.items():
    print(f"  {name}: {reps.shape}")

In [ ]:
# Apply cross-model RSA
rsa_models = CrossSpeciesRSA()
model_similarity_matrix = rsa_models.compute_similarity_matrix(model_reps)

# Compare to biological similarity
# Use subset of biological species for comparison
bio_subset = {name: species_reps[name] for name in ['Mouse', 'Macaque', 'Human']}
bio_similarity_matrix = rsa_models.compute_similarity_matrix(bio_subset)

print("\n✓ Cross-model similarity computed")
print(f"  Model similarity matrix: {model_similarity_matrix.shape}")
print(f"  Bio similarity matrix: {bio_similarity_matrix.shape}")

In [ ]:
# Interactive comparison: Models vs Biology

dataset_names = list(model_reps.keys())

# Panel 1: Model similarity matrix
p_model_sim = figure(
    title='Foundation Model Similarity (Trained on Different Datasets)',
    width=500,
    height=450,
    x_range=dataset_names,
    y_range=list(reversed(dataset_names)),
    toolbar_location=None
)

# Flatten
model_x = []
model_y = []
model_sim_values = []

for i, d1 in enumerate(dataset_names):
    for j, d2 in enumerate(dataset_names):
        model_x.append(d1)
        model_y.append(d2)
        model_sim_values.append(model_similarity_matrix[i, j])

model_sim_mapper = LinearColorMapper(palette=RdBu11[::-1], 
                                    low=model_similarity_matrix.min(), 
                                    high=model_similarity_matrix.max())

p_model_sim.rect(x=model_x, y=model_y, width=1, height=1,
                fill_color={'field': 'sim', 'transform': model_sim_mapper},
                line_color='white', line_width=2,
                source={'x': model_x, 'y': model_y, 'sim': model_sim_values})

# Add text
source_model = ColumnDataSource(data={'x': model_x, 'y': model_y, 
                                      'text': [f'{s:.2f}' for s in model_sim_values]})
labels_model = LabelSet(x='x', y='y', text='text', source=source_model,
                       text_align='center', text_baseline='middle', text_font_size='9pt')
p_model_sim.add_layout(labels_model)

color_bar_model = ColorBar(color_mapper=model_sim_mapper, width=15, location=(0, 0),
                          title='Similarity')
p_model_sim.add_layout(color_bar_model, 'right')
p_model_sim.xaxis.major_label_orientation = 0.785

# Panel 2: Biological similarity matrix (subset)
bio_names_subset = list(bio_subset.keys())

p_bio_sim = figure(
    title='Biological Species Similarity',
    width=400,
    height=350,
    x_range=bio_names_subset,
    y_range=list(reversed(bio_names_subset)),
    toolbar_location=None
)

# Flatten
bio_x = []
bio_y = []
bio_sim_values = []

for i, s1 in enumerate(bio_names_subset):
    for j, s2 in enumerate(bio_names_subset):
        bio_x.append(s1)
        bio_y.append(s2)
        bio_sim_values.append(bio_similarity_matrix[i, j])

bio_sim_mapper = LinearColorMapper(palette=RdBu11[::-1], 
                                  low=bio_similarity_matrix.min(), 
                                  high=bio_similarity_matrix.max())

p_bio_sim.rect(x=bio_x, y=bio_y, width=1, height=1,
              fill_color={'field': 'sim', 'transform': bio_sim_mapper},
              line_color='white', line_width=2,
              source={'x': bio_x, 'y': bio_y, 'sim': bio_sim_values})

# Add text
source_bio = ColumnDataSource(data={'x': bio_x, 'y': bio_y, 
                                    'text': [f'{s:.2f}' for s in bio_sim_values]})
labels_bio = LabelSet(x='x', y='y', text='text', source=source_bio,
                     text_align='center', text_baseline='middle', text_font_size='9pt')
p_bio_sim.add_layout(labels_bio)

color_bar_bio = ColorBar(color_mapper=bio_sim_mapper, width=15, location=(0, 0),
                        title='Similarity')
p_bio_sim.add_layout(color_bar_bio, 'right')

# Combine
grid_compare = gridplot([[p_model_sim, p_bio_sim]])
show(grid_compare)

print("\n📊 Models vs Biology Comparison")
print("   Left: Foundation models trained on different datasets")
print("   Right: Biological species (evolutionarily related)")
print("   \n   Question: Do models show similar patterns of similarity?")

# Quantify similarity structure
model_sim_std = model_similarity_matrix[np.triu_indices(len(dataset_names), k=1)].std()
bio_sim_std = bio_similarity_matrix[np.triu_indices(len(bio_names_subset), k=1)].std()

print(f"\n   Model similarity variability: {model_sim_std:.3f}")
print(f"   Biological similarity variability: {bio_sim_std:.3f}")
print(f"   \n   {'Models are MORE diverse' if model_sim_std > bio_sim_std else 'Biology is MORE diverse'}")

---

## Exercises

1. **Null models**: Generate random representations and test whether observed phylogenetic correlation is significant compared to random.

2. **Layer-wise analysis**: For deep networks, apply cross-species methods to each layer. Does alignment improve in higher layers?

3. **Task specificity**: Train models on different tasks (classification, segmentation, detection). Does task induce structure similar to species differences?

4. **Alignment algorithms**: Compare Procrustes to other alignment methods (CCA, optimal transport). Which best captures biological similarity?

5. **Functional annotation**: For conserved components, test which stimulus features they encode (orientation, color, motion, etc.).

---

## Summary

This notebook demonstrated:

1. **Procrustes Alignment**: Optimal transformations between neural spaces
2. **RSA**: Geometry-based comparison using RDMs
3. **Phylogenetic Analysis**: Linking neural similarity to evolutionary distance
4. **CSD**: Decomposing representations into conserved vs specific components
5. **Foundation Model Testing**: Applying evolutionary methods to AI

**Key Takeaways**:

- **Cross-species methods** reveal universal computational principles
- **Phylogenetic constraints** shape neural representations
- **Conserved features** likely solve fundamental problems (e.g., edge detection)
- **Species-specific features** reflect ecological adaptations
- **Foundation models** can be evaluated using the same comparative framework

**Philosophical Implications**:

Evolution discovered optimal solutions to sensory processing problems over millions of years. If foundation models converge to similar solutions through gradient descent, this suggests:

1. These solutions are **functionally optimal** (not arbitrary)
2. The **problem structure** constrains the solution space
3. Different optimization algorithms (evolution vs SGD) can reach the same point

This is evidence for a **normative theory** of neural computation.

---

### References

1. Kriegeskorte, N., Mur, M., & Bandettini, P. (2008). Representational similarity analysis. *Frontiers in Systems Neuroscience*, 2, 4.

2. Gower, J. C., & Dijksterhuis, G. B. (2004). *Procrustes Problems*. Oxford University Press.

3. Yamins, D. L., & DiCarlo, J. J. (2016). Using goal-driven deep learning models to understand sensory cortex. *Nature Neuroscience*, 19(3), 356-365.

4. Kriegeskorte, N., & Wei, X. X. (2021). Neural tuning and representational geometry. *Nature Reviews Neuroscience*, 22(11), 703-718.

5. Gao, P., & Ganguli, S. (2015). On simplicity and complexity in the brave new world of large-scale neuroscience. *Current Opinion in Neurobiology*, 32, 148-155.

---

*Created with NeuroS-MechInt • Advanced Mechanistic Interpretability for Neuroscience Foundation Models*